# Classify all systems with the neural-network models (leaflet-split training)

Uses the models saved by `NN_training` (`models_scalers/nn_{chain}_{n}pt_model_ft_last.sav` + scaler)
through `clf_lo_ld_by_chain` in `predict_acyl_chains_NN_by_chain.py` (the same class the NN uses).

- Set **`N_POINT`** to the window you chose from `NN_acc_ece` (or `"raw"`).
- Systems: `dopc_pure` (χ_C = 0), the mixed systems `dopc_94` … `dopc_65`, `dopc_60` (χ_C = 0.40) and the two
  χ_C = 0.25 replicas `dopc_75_rep1`, `dopc_75_rep2` (300 DOPC per leaflet, so they need about twice the memory).
  `dopc_pure` has no cholesterol, so its CHOL arrays are saved empty.
- Both leaflets of every system are classified. Features are built exactly as in training:
  lipid-major rows, n-pt running average of each lipid over time.
- `prediction_probs` = P(Lo), the network output; `prediction_labels` = P(Lo) > 0.5.
- Arrays are (time, lipid) and line up with `acyl_coms` (time, lipid, 3). Each file holds (lower, upper).

Output: `heavy_data_NO_GIT/data_arrays_by_chain_nn/{system}_{chain}_ft_last_{n}pt_{quantity}.npy`
(`raw` window: `{system}_{chain}_raw_{quantity}.npy`), one folder per model so LR and XGB don't overwrite each other.

## 1. Settings

In [ ]:
import sys
sys.path.insert(0, "..")          # code_libraries is in the repository root; data in ../heavy_data_NO_GIT

import os
import time
import numpy as np

from code_libraries.predict_acyl_chains_NN_by_chain import clf_lo_ld_by_chain

MODEL_TYPE = "nn"
N_POINT    = 5                  # window used for training/choosing the model, or "raw"

start_frame, stop_frame = 0, 4000
sys_type = "sym"
SYSTEMS  = ["dopc_pure", "dopc_94", "dopc_90", "dopc_85", "dopc_80", "dopc_75", "dopc_70", "dopc_65", "dopc_60",
            "dopc_75_rep1", "dopc_75_rep2"]
CHAINS   = ["sn1", "sn2"]

data_array_dir = os.path.join("../", "heavy_data_NO_GIT/data_arrays_by_chain_nn")
os.makedirs(data_array_dir, exist_ok=True)

if N_POINT == "raw":
    red_type, n_point_arg, file_tag = "raw", 1, "raw"          # n_point = 1 -> no frames lost
else:
    red_type, n_point_arg, file_tag = "ft_last", int(N_POINT), f"ft_last_{int(N_POINT)}pt"

## 2. Classify and save, one system at a time

In [ ]:
QUANTITIES = {"box_dim":            lambda clf: clf.box_dimensions,
              "acyl_coms":          lambda clf: clf.processed_acyl_coms,
              "acyl_c4":            lambda clf: clf.processed_acyl_c4,
              "chol_coms":          lambda clf: clf.processed_chol_coms,
              "chol_heads":         lambda clf: clf.processed_chol_heads,
              "prediction_probs":   lambda clf: clf.prediction_probs,
              "prediction_labels":  lambda clf: clf.prediction_labels}

t_all = time.time()
for i, sys_name in enumerate(SYSTEMS, start=1):
    for chain in CHAINS:
        t = time.time()
        clf = clf_lo_ld_by_chain(sys_name, sys_type, red_type, MODEL_TYPE, chain, n_point_arg,
                                 start_frame, stop_frame)
        clf.process_all_input()
        clf.classify(uses_predict_proba=False)         # the network output is already P(Lo)

        probs_lower, probs_upper = clf.prediction_probs
        assert probs_lower.min() >= 0 and probs_lower.max() <= 1
        assert probs_lower.shape == clf.processed_acyl_coms[0].shape[:2], "predictions do not line up with acyl COMs"

        for name, get in QUANTITIES.items():
            np.save(os.path.join(data_array_dir, f"{sys_name}_{chain}_{file_tag}_{name}.npy"), get(clf))

        lo_lower, lo_upper = (p.mean() for p in clf.prediction_labels)
        print(f"[{i}/{len(SYSTEMS)}] {sys_name} {chain}: {probs_lower.shape[0]} frames x {probs_lower.shape[1]} lipids, "
              f"Lo fraction lower {lo_lower:.3f}, upper {lo_upper:.3f}  ({time.time() - t:.0f} s)", flush=True)
        del clf

print(f"\nall systems done in {(time.time() - t_all) / 60:.1f} min")